In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -qq -U evaluate rouge_score

In [ ]:
from unsloth import FastLanguageModel
import torch  
import time 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Rohit-Katkar2003/qwen-0.6b-fine-tune-4",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

def generate_response(prompt): 

    start_time = time.time()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1, 
        repetition_penalty=1.2,   # 🔥 ADD THIS
        no_repeat_ngram_size=3,   # 🔥 ADD THIS
        top_p=0.9
    )
    fresponse = tokenizer.decode(outputs[0], skip_special_tokens=True)  
    response = fresponse[len(prompt):].strip()    ## it write question in every response 

    end_time = time.time()
    return response , round(end_time - start_time , 2)
    
print(generate_response("hi"))

==((====))==  Unsloth 2026.3.5: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen3-0.6B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
(". I'm trying to find the value of $ \\frac{1}{2} + 3\\left( \\frac{\\pi^4}{60} - \\frac{(5/8)^4}{4!} \\right) $. Can you help me? \n\nI know that $\\frac{\\Gamma(n)}{n}$ is equal to a factorial, but how do we get there?\n\nLet's say n = k and m = l.\n\nSo for example, if we have something like $\\frac{k(k-1)(k-2)...(a-b+1)}{b-a+1}$ then it can be", 7.79)


### Model Testing

In [7]:
test_cases = [
    {
        "type": "instruction",
        "prompt": "Translate this to French: 'Hello, how are you?'",
    },
    {
        "type": "summarization",
        "prompt": "Summarize: Artificial Intelligence is transforming industries by automating tasks, improving decision-making, and enabling new products.",
    },
    {
        "type": "extraction",
        "prompt": "Extract name and age: John is 28 years old and lives in New York.",
    },
    {
        "type": "decomposition",
        "prompt": "Break this problem into steps: Build a machine learning pipeline.",
    },
    {
        "type": "reasoning",
        "prompt": "If a train travels 60 km in 1 hour, how far in 3 hours?",
    },
    {
        "type": "hallucination",
        "prompt": "Who is the president of Mars?",
    }
] 


results = []

for test in test_cases:
    print("\n=========================")
    print(f"Test Type: {test['type']}")
    print("Prompt:", test["prompt"])

    response, latency = generate_response(test["prompt"])

    print("Response:", response)
    print("Latency:", latency, "sec")

    results.append({
        "type": test["type"],
        "response": response,
        "latency": latency
    })


Test Type: instruction
Prompt: Translate this to French: 'Hello, how are you?'
Response: 'Bonjour, comment ça va ?'
The translation of the phrase "Hello, How are you?" into French is indeed "Bonjour, Comment ça va ?", which means “Good morning / Good day” in English. The word order and structure follow a similar pattern as in French language.
In summary, translating "Hello" would be "Bienvenue", while "how are you" translates to "Comment ça va". Therefore, the complete sentence becomes "Bienvenu, comment ca va?"
This response highlights that the translation follows the same grammatical structure as in the original language, making it easier for non-native speakers to understand and use
Latency: 7.81 sec

Test Type: summarization
Prompt: Summarize: Artificial Intelligence is transforming industries by automating tasks, improving decision-making, and enabling new products.
Response: However, there are also risks associated with AI technology such as bias in algorithms, privacy concerns,